# Module 1: Python Intermediate

**EJ** · Nov 3–6, 2026 · image: python-training:1.0

## Environment Check

Shows the Python version used by Jupyter, verifies the shared folders and what's inside them, and checks whether the environment variables exist.

In [1]:
import sys
import os
print(f'Python version: {sys.version}')

Python version: 3.11.18 (main, Oct 28 2026, 02:02:50) [GCC 14.2.0]


In [2]:
print("Shared folders and what's inside them")
!ls /workspace/work /workspace/data

Shared folders and what's inside them
/workspace/data:
module1

/workspace/work:
__pycache__  config.py	docs  module1_python_intermediate.ipynb  scratch


In [3]:
print('Variables from .env: ')
print(f'Token exists: {"JUPYTER_TOKEN" in os.environ}, Port exists: {"JUPYTER_PORT" in os.environ}')

Variables from .env: 
Token exists: True, Port exists: True


## 1.1 Comprehensions

In [4]:
from config import DATA_ROOT

DATA_DIR = DATA_ROOT / "module1"

### B-1.1.1 Clean the amount strings

Build a cleaned list of floats from 20 messy amount strings with one comprehension that strips, converts, and drops values at or below zero (Lab 1.1 step 1).

In [5]:
amounts_file = DATA_DIR / "amounts_raw.txt"

# one long string, .splitlines() cuts the string at each line break and creates a list
amounts = amounts_file.read_text().splitlines()

# The generator cleans and transforms each amount when the loop asks for it, the result is then filtered to keep the right values
# Transform -> Filter: the transform happens in the iterable itself, not on the left side
clean_amounts = [value for value in (float(amount.strip()) for amount in amounts) if value > 0]
print(f'Clean amounts: {clean_amounts}')
print(f'{len(clean_amounts)} of {len(amounts)} kept')

Clean amounts: [90.85, 51.1, 32.03, 10.64, 82.48, 48.97, 45.39, 80.18, 48.59, 91.31, 55.67, 77.94, 89.91, 26.49, 51.49, 68.55, 81.97]
17 of 20 kept


### B-1.1.2 Same logic as a for loop

Rewrite the same logic as a plain `for` loop and compare readability (Lab 1.1 step 2).

In [6]:
clean_amounts_loop = []

for amount in amounts:
    value = float(amount.strip())
    if value > 0:
        clean_amounts_loop.append(value)

print(f'Clean amounts (loop): {clean_amounts_loop}')
print(f'Comprehension and Loop created the same ouput: {clean_amounts == clean_amounts_loop}')
print(f'{len(clean_amounts_loop)} of {len(amounts)} kept')

Clean amounts (loop): [90.85, 51.1, 32.03, 10.64, 82.48, 48.97, 45.39, 80.18, 48.59, 91.31, 55.67, 77.94, 89.91, 26.49, 51.49, 68.55, 81.97]
Comprehension and Loop created the same ouput: True
17 of 20 kept


**Readability**
In terms of readability, the loop is easier to read here because each step is on its own line in sequential order, while a comprehension compresses the same logic into one line: it's shorter, but harder to follow.

**When a loop is a better choice?**
A loop is a better choice when the logic is more complex (needs several steps), has side effects like printing, or needs error handling because the more complex the logic, the harder it is to read in a comprehension.

### B-1.1.3 Total quantity per product (dict comprehension)

Build a dictionary comprehension mapping product name to total quantity (Lab 1.1 step 3).

In [7]:
import csv
with open(DATA_DIR / "orders.csv", newline="") as f:
    # csv.DictReader is a lazy loader, so list() catches all the rows before the file closes
    orders = list(csv.DictReader(f))
    
product_quantity = {
    # each value sums only that product's rows
    product: sum(int(row["quantity"]) for row in orders 
                 # isdigit() skips invalid quantities like x or -1
                 if row["quantity"].isdigit() and row["product"] == product) 
    for product in {row["product"] for row in orders}}

print(f'Total Quantity per product: {product_quantity}')

Total Quantity per product: {'Notebook': 25, 'Folder': 10, 'Glue': 7, 'Eraser': 62, 'Pen': 11, 'Highlighter': 54, 'Stapler': 14, 'Marker': 32, 'Ruler': 15}


In [8]:
# Here is the loop version of the comprehension logic above

product_quantity_loop = {}

for row in orders:
    product_name = row["product"]
    if row["quantity"].isdigit():
        product_quantity_loop[product_name] = int(row["quantity"]) + product_quantity_loop.get(product_name, 0)

print(f'Total Quantity per product (loop): {product_quantity_loop}')

Total Quantity per product (loop): {'Eraser': 62, 'Notebook': 25, 'Marker': 32, 'Stapler': 14, 'Ruler': 15, 'Highlighter': 54, 'Pen': 11, 'Folder': 10, 'Glue': 7}


**What a comprehension does that a loop doesn't**

A comprehension creates and fills a collection in one go, it doesn't have a concept of an empty container and an append logic that a loop needs. It also has a compact way of filtering and transforming in one expression, right side for filtering and left side for transforming.

An example from my code (B-1.1.3) shows that my loop kept a running total for each product with `.get(product_name, 0) + product_quantity`. My comprehension couldn't do that, because it has no way to update a value it already wrote as each key is written with its final value, so instead, it used sum() over the matching rows for each unique product.

## 1.2 Packages and Environments

### B-1.2.1 Importing

Import one standard library module and one third-party package in the notebook (Lab 1.2 step 1).


In [9]:
import random # standard library
print(f'{random.random()}: prints a random float between 0.0 and 1.0')

import tabulate # third-party package, raises a ModuleNotFoundError because we haven't installed it yet

0.1433456662106174: prints a random float between 0.0 and 1.0


### B-1.2.2 Pinning a third-party package

Add the third-party package to `requirements.txt` with a pinned version, rebuild the image, restart the container, and confirm the import works (Lab 1.2 step 2).

In [10]:
from importlib.metadata import version
from tabulate import tabulate

# Uses the specific version in the requirements.txt
print(f'tabulate {version("tabulate")} (from the rebuilt image)\n')
print(tabulate([["Pen", 11], ["Glue", 7]], headers=["product", "quantity"]))

tabulate 0.9.0 (from the rebuilt image)

product      quantity
---------  ----------
Pen                11
Glue                7


### B-1.2.3 Creating a Virtual Environment

Create a virtual environment in a scratch folder, install a different version of the same package, and show the two environments report different versions (Lab 1.2 step 3).

! at the start tells jupyter to run this as a shell command, not python code

In [11]:
!python -m venv /tmp/venv-old 
# creates a new environment, the /tmp/venv-old will get its own copy of python and its own packages folder

In [12]:
!/tmp/venv-old/bin/pip install -q tabulate==0.8.10 
# installs a specific version of tabulate in the new environment


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip


In [13]:
# Different version for each environment

image_version = !pip show tabulate | grep Version
venv_version = !/tmp/venv-old/bin/pip show tabulate | grep Version
venv_requirements = !cat /tmp/requirements.txt

print(f"Image {image_version[0]}")
print(f"Image requirements.txt {venv_requirements}")
print(f"Venv {venv_version[0]}")


Image Version: 0.9.0
Image requirements.txt ['jupyterlab==4.4.0', 'tabulate==0.9.0']
Venv Version: 0.8.10


#### B-1.2.4 When a virtual environment is needed inside a container

A virtual environment inside a container is useful when two pieces of code need different versions of the same package side by side (such as during code migration). Another specific scenario seen in the labs is when you want to quickly try a different version without rebuilding the image. 

**When an image alone is enough**
For a single project with one set of dependencies, the image alone is enough since the container already provides isolation, and anyone who builds the same image will get the specific package versions pinned in requirements.txt.

## 1.3 Exceptions

### B-1.3.1 Create a safe_to_float function

Write `safe_to_float(value)` that returns a float or raises a clear message for bad input (Lab 1.3 step 1).

In [14]:
def safe_to_float(value) -> float:
    try:
        result = float(value)
    except ValueError as err:
        raise ValueError(f'{value!r} cannot be converted into a float') from err
    except TypeError as err:
        raise TypeError(f'Expected a number or string, got {type(value).__name__}') from err
    else:
        return result

### B-1.3.2 + B-1.3.3: Counting errors, with else and finally

Feed it valid numbers, empty strings, letters, and `None`; catch specific exceptions and count each type (Lab 1.3 step 2).
Demonstrate `else` and `finally` with a real example (Lab 1.3 step 3).

In [22]:
import json

mixed_values = json.loads((DATA_DIR / "values_mixed.json").read_text())
valid_values = []
counts_per_outcome = {"valid": 0, "ValueError": 0, "TypeError": 0}

for value in mixed_values:
    try:
        result = safe_to_float(value)
    except ValueError as err:
        counts_per_outcome["ValueError"] += 1
        print(f"Skipped: {err}")
    except TypeError as err:
        counts_per_outcome["TypeError"] += 1
        print(f"Skipped: {err}")
    else:
        valid_values.append(result)
        counts_per_outcome["valid"] += 1
    finally:
        print(f"Processed: {value!r}")

print(f'\nValid values {valid_values}')
print(f'Counts per outcome {counts_per_outcome}')

Processed: 12.5
Processed: '7'
Skipped: '' cannot be converted into a float
Processed: ''
Skipped: 'abc' cannot be converted into a float
Processed: 'abc'
Skipped: Expected a number or string, got NoneType
Processed: None
Processed: '3.14'
Processed: ' 42 '
Skipped: 'N/A' cannot be converted into a float
Processed: 'N/A'
Processed: 0
Processed: '-5'
Processed: '1e3'
Skipped: '' cannot be converted into a float
Processed: ''
Skipped: Expected a number or string, got NoneType
Processed: None
Skipped: 'hello' cannot be converted into a float
Processed: 'hello'
Processed: 99

Valid values [12.5, 7.0, 3.14, 42.0, 0.0, -5.0, 1000.0, 99.0]
Counts per outcome {'valid': 8, 'ValueError': 5, 'TypeError': 2}


The loop above shows both: else appends only successful conversions; finally counts every value processed.

#### else vs finally, and why catch specific exceptions

An else block runs only when the try block didn’t fail, while a finally block runs whether or not the try block failed. 

Catching specific exceptions like ValueError and TypeError is better than using a bare except because it allows the program to handle expected data errors without stopping the entire process, while still allowing unexpected errors caused by programming bugs to surface instead of being silently ignored.

## What broke and how I fixed it

## Reflection